# Optional Indecisive Ranker Training

This notebook trains a tiny optional ranker and exports `ranker.pkl` plus `ranker_metadata.json`. FastAPI never trains models; it only loads copied artifacts when `ENABLE_TRAINED_RANKER=true`.

In [ ]:
!pip install -q scikit-learn joblib numpy

In [ ]:
import json
from datetime import datetime, timezone
from pathlib import Path

import joblib
import numpy as np
from sklearn.ensemble import RandomForestRegressor

rng = np.random.default_rng(42)
base_score = rng.uniform(0.1, 0.95, size=500)
distance_score = rng.uniform(0.0, 1.0, size=500)
semantic_score = rng.uniform(0.2, 0.95, size=500)
X = np.column_stack([base_score, distance_score, semantic_score])
y = (0.55 * base_score) + (0.25 * distance_score) + (0.20 * semantic_score)

model = RandomForestRegressor(n_estimators=50, random_state=42, max_depth=5)
model.fit(X, y)

exports = Path('/content/indecisive_exports')
exports.mkdir(exist_ok=True)
joblib.dump(model, exports / 'ranker.pkl')
(exports / 'ranker_metadata.json').write_text(json.dumps({
    'trained_at': datetime.now(timezone.utc).isoformat(),
    'feature_order': ['base_score', 'distance_score', 'semantic_score'],
    'source': 'synthetic optional Colab demo data'
}, indent=2))
exports

Download the two files from `/content/indecisive_exports` and copy them into the local repo's `models/` directory. Keep `ENABLE_TRAINED_RANKER=false` until you intentionally want to test the trained ranker.